In [247]:
# Prepare data
import numpy as np

data_root = "./timit_11"
train = np.load(data_root + "/train_11.npy")
train_label = np.load(data_root + "/train_label_11.npy")
test = np.load(data_root + "/test_11.npy")

print(f"Size of training data: {train.shape}")
print(f"Size of testing data: {test.shape}")

Size of training data: (1229932, 429)
Size of testing data: (451552, 429)


In [248]:
# Create Dataset
import torch
from torch.utils.data import Dataset

class TIMITDataset(Dataset):
    def __init__(self, X, y=None):
        self.data = torch.from_numpy(X).float()
        if y is not None: # train
            y = y.astype(int)
            self.label = torch.LongTensor(y)
        else: # test
            self.label = None
            
    def __getitem__(self, idx):
        if self.label is not None:
            return self.data[idx], self.label[idx]
        else:
            return self.data[idx]
        
    def __len__(self):
        return len(self.data)

In [249]:
VAL_RATIO = 0.2 # Ratio of validation_set to the hole training data

percent = int(train.shape[0] * (1 - VAL_RATIO))
train_x, train_y, val_x, val_y = train[:percent], train_label[:percent], train[percent:], train_label[percent:]

print(f"Size of training set: {train_x.shape}")
print(f"Size of validation set: {val_x.shape}")

Size of training set: (983945, 429)
Size of validation set: (245987, 429)


In [250]:
# Prepare dataloader
BATCH_SIZE = 256

from torch.utils.data import DataLoader

train_set = TIMITDataset(train_x, train_y)
val_set = TIMITDataset(val_x, val_y)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

In [251]:
# Optional: garbage collection
import gc

del train, train_label, train_x, train_y, val_x, val_y
gc.collect() # Execute garbage collection

84

In [252]:
# Define model
import torch
import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        self.layer1 = nn.Linear(429, 2048)
        self.batch_norm_1 = nn.BatchNorm1d(2048)
        self.layer2 = nn.Linear(2048, 1024)
        self.batch_norm_2 = nn.BatchNorm1d(1024)
        self.layer3 = nn.Linear(1024, 512)
        self.batch_norm_3 = nn.BatchNorm1d(512)
        self.layer4 = nn.Linear(512, 256)
        self.batch_norm_4 = nn.BatchNorm1d(256)
        self.layer5 = nn.Linear(256, 128)
        self.batch_norm_5 = nn.BatchNorm1d(128)
        self.out = nn.Linear(128, 39)
        
        self.act_fn = nn.ReLU() # ReLU can help prevent the vanishing gradient problem
        
        self.dropout1 = nn.Dropout(0.5)
        self.dropout2 = nn.Dropout(0.4)
        self.dropout3 = nn.Dropout(0.3)
        self.dropout4 = nn.Dropout(0.2)
        self.dropout5 = nn.Dropout(0.1)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.batch_norm_1(x)
        x = self.act_fn(x)
        x = self.dropout1(x)
        
        x = self.layer2(x)
        x = self.batch_norm_2(x)
        x = self.act_fn(x)
        x = self.dropout2(x)
        
        x = self.layer3(x)
        x = self.batch_norm_3(x)
        x = self.act_fn(x)
        x = self.dropout3(x)
        
        x = self.layer4(x)
        x = self.batch_norm_4(x)
        x = self.act_fn(x)
        x = self.dropout4(x)
        
        x = self.layer5(x)
        x = self.batch_norm_5(x)
        x = self.act_fn(x)
        x = self.dropout5(x)
        
        x = self.out(x)
        
        return x

In [253]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    else:
        return "cpu"

In [254]:
# Fix a random seed
def fix_seeds(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [255]:
# Training parameters
fix_seeds(0)

device = get_device()
print(f"DEVICE: {device}")

num_epochs = 100
learning_rate = 0.0001
l2_lambda = 1e-4

model_path = "./model.ckpt"

# Crate a model and define a loss function and optimizer
model = Classifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=l2_lambda)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=5)

DEVICE: cuda


In [256]:
# Training start!
best_acc = 0.0 # Accuracy
for epoch in range(num_epochs):
    train_acc = 0.0
    train_loss = 0.0
    val_acc = 0.0
    val_loss = 0.0
    
    model.train()
    for i, data in enumerate(train_loader):
        samples, labels = data
        samples, labels = samples.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(samples)
        batch_loss = criterion(outputs, labels)
        _, train_preds = torch.max(outputs, 1) # Get the index of the class with the highest probability, which is regarded as the predicted class
        batch_loss.backward()
        optimizer.step()
        
        train_acc += (train_preds.cpu() == labels.cpu()).sum().item()
        train_loss += batch_loss.item()
        
    # Validation
    if len(val_set) > 0:
        model.eval()
        with torch.no_grad():
            for i, data in enumerate(val_loader):
                samples, labels = data
                samples, labels = samples.to(device), labels.to(device)
                outputs = model(samples)
                batch_loss = criterion(outputs, labels)
                _, val_preds = torch.max(outputs, 1)
                
                val_acc += (val_preds.cpu() == labels.cpu()).sum().item()
                val_loss += batch_loss.item()
            
            print("[{:03d}/{:03d}] Train Acc: {:3.6f} Loss: {:3.6f} | Val Acc: {:3.6f} loss: {:3.6f}".format(
                epoch + 1, num_epochs, train_acc/len(train_set), train_loss/len(train_loader), val_acc/len(val_set), val_loss/len(val_loader)
            ))
            
            avg_val_loss = val_loss / len(val_loader)
            scheduler.step(avg_val_loss)
            
            # If the model has improved, then save it
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(model.state_dict(), model_path)
                print(f"Save the model with acc: {best_acc/len(val_set):.3f}")
                
    else:
        print("[{:03d}/{:03d}] Train Acc: {:3.6f} Loss: {:3.6f}".format(
            epoch + 1, num_epochs, train_acc/len(train_set), train_loss/len(train_loader)
        ))
        
# If without validation, save the result of the last epoch
if len(val_set) == 0:
    torch.save(model.state_dict(), model_path)
    print("Save the model at the last epoch")

[001/100] Train Acc: 0.513711 Loss: 1.681909 | Val Acc: 0.648957 loss: 1.138813
Save the model with acc: 0.649
[002/100] Train Acc: 0.602027 Loss: 1.295797 | Val Acc: 0.676893 loss: 1.025562
Save the model with acc: 0.677
[003/100] Train Acc: 0.625542 Loss: 1.205241 | Val Acc: 0.693281 loss: 0.965347
Save the model with acc: 0.693
[004/100] Train Acc: 0.640385 Loss: 1.149771 | Val Acc: 0.703679 loss: 0.925230
Save the model with acc: 0.704
[005/100] Train Acc: 0.650431 Loss: 1.112531 | Val Acc: 0.709294 loss: 0.902296
Save the model with acc: 0.709
[006/100] Train Acc: 0.658356 Loss: 1.081139 | Val Acc: 0.714208 loss: 0.882049
Save the model with acc: 0.714
[007/100] Train Acc: 0.665090 Loss: 1.056577 | Val Acc: 0.718717 loss: 0.866738
Save the model with acc: 0.719
[008/100] Train Acc: 0.671522 Loss: 1.033180 | Val Acc: 0.722189 loss: 0.852386
Save the model with acc: 0.722
[009/100] Train Acc: 0.676084 Loss: 1.016424 | Val Acc: 0.725087 loss: 0.840430
Save the model with acc: 0.725
[

In [257]:
# Prepare for testing
test_set = TIMITDataset(test, None)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

model = Classifier().to(device)
model.load_state_dict(torch.load(model_path))

<All keys matched successfully>

In [258]:
predictions = []
model.eval()
with torch.no_grad():
    for i, data in enumerate(test_loader):
        samples = data
        samples = samples.to(device)
        outputs = model(samples)
        _, test_preds = torch.max(outputs, 1)
        
        for y in test_preds.cpu().numpy():
            predictions.append(y)

In [259]:
with open("prediction.csv", 'w') as f:
    f.write("Id,Class\n")
    for i, y in enumerate(predictions):
        f.write(f"{i},{y}\n")